## <span style="color: teal;">Merging and Preprocessing Environmental Data for Machine Learning </span>

This notebook processes and merges various environmental datasets, including MODIS satellite data, weather stations, and topographic variables, to create a comprehensive dataset for machine learning. It aggregates and cleans values for parameters such as temperature, precipitation, wind, and watewr vapor column. The resulting final dataframe is prepared for training machine learning models to analyze and predict environmental trends.

### <span style="color: teal;">Libraries</span>

In [1]:
import os
import pandas as pd
import numpy as np
from datetime import datetime

### <span style="color: teal;">Load MODIS data & Define Function to Process Nearest Values</span>

In [68]:
# Function to ensure the directory exists
def ensure_directory_exists(directory):
    if not os.path.exists(directory):
        os.makedirs(directory)

# Function to process nearest values
def process_nearest_values(file_path, value_column, output_file):
    """Process the nearest pixel values for a given dataset and save cleaned results."""
    df = pd.read_csv(file_path)
    
    # Extract the Date from 'Tif Name' (format: .AYYYYDDD)
    df['Date'] = df['Tif Name'].str.extract(r'\.A(\d{7})')[0]
    df['Date'] = pd.to_datetime(df['Date'], format='%Y%j')  # Convert YYYYDDD to datetime

    # Aggregate by Date and Station Name
    aggregated_data = []
    for (date, station_name), group in df.groupby(['Date', 'Station Name']):
        pixel_values = group[value_column].values
        aggregated_value = np.nanmean(pixel_values)  # Compute mean (ignoring NaNs!!!)

        aggregated_row = group.iloc[0].copy()
        aggregated_row[value_column] = aggregated_value
        aggregated_row['Tif Name'] = f'{date.strftime("%Y-%m-%d")}_{station_name.replace(" ", "_")}.tif'

        aggregated_data.append(aggregated_row)

    # Create a new DataFrame
    aggregated_df = pd.DataFrame(aggregated_data)

    # Reset index to start from 1
    aggregated_df.index = range(1, len(aggregated_df) + 1)

    # Remove rows where value_column is NaN
    aggregated_df_no_nans = aggregated_df.dropna(subset=[value_column]).reset_index(drop=True)

    # Ensure output directory exists
    output_dir = os.path.dirname(output_file)
    ensure_directory_exists(output_dir)

    # Save cleaned data to CSV
    aggregated_df_no_nans.to_csv(output_file, index=False)

    return aggregated_df_no_nans


# Process all three datasets and save cleaned CSVs
aggregated_df_no_nans_CWP = process_nearest_values("Nearest_pixels_to_stations4/CWP/nearest_values_CWP.csv", 'Value CWP', "Nearest_pixels_to_stations5/CWP/CWP_aggregated_data_no_nans.csv")
aggregated_df_no_nans_CER = process_nearest_values("Nearest_pixels_to_stations4/CER/nearest_values_CER.csv", 'Value CER', "Nearest_pixels_to_stations5/CER/CER_aggregated_data_no_nans.csv")
aggregated_df_no_nans_COT = process_nearest_values("Nearest_pixels_to_stations4/COT/nearest_values_COT.csv", 'Value COT', "Nearest_pixels_to_stations5/COT/COT_aggregated_data_no_nans.csv")

/tmp/ipykernel_15901/946447715.py:19: RuntimeWarning: Mean of empty slice
  aggregated_value = np.nanmean(pixel_values)  # Compute mean (ignoring NaNs!!!)
/tmp/ipykernel_15901/946447715.py:19: RuntimeWarning: Mean of empty slice
  aggregated_value = np.nanmean(pixel_values)  # Compute mean (ignoring NaNs!!!)
/tmp/ipykernel_15901/946447715.py:19: RuntimeWarning: Mean of empty slice
  aggregated_value = np.nanmean(pixel_values)  # Compute mean (ignoring NaNs!!!)


### <span style="color: teal;">Load All Processed Datasets <span>

In [69]:
# Load all processed datasets
aggregated_df_CWP_no_nans = pd.read_csv("Nearest_pixels_to_stations4/CWP/CWP_aggregated_data_no_nans.csv")
aggregated_df_CER_no_nans = pd.read_csv("Nearest_pixels_to_stations4/CER/CER_aggregated_data_no_nans.csv")
aggregated_df_COT_no_nans = pd.read_csv("Nearest_pixels_to_stations4/COT/COT_aggregated_data_no_nans.csv")
wind_df = pd.read_csv("Nearest_pixels_to_stations4/Wind/nearest_values_Wind.csv")
daily_station_summed_IMERG = pd.read_csv("imerg_station_nearest_values/imerg_nearest_values.csv")
water_vapor_df = pd.read_csv("Nearest_pixels_to_stations4/Water_Vapor_Column/nearest_values_Water_Vapor_Column.csv")
aspect_df = pd.read_csv("Nearest_pixels_to_stations4/ASPECT/nearest_values_ASPECT.csv")
lst_df = pd.read_csv("Nearest_pixels_to_stations4/LST/nearest_values_LST.csv")
slope_df = pd.read_csv("Nearest_pixels_to_stations4/SLOPE/nearest_values_SLOPE.csv")
tpi_df = pd.read_csv("Nearest_pixels_to_stations4/TPI/nearest_values_TPI.csv")
df_twi = pd.read_csv("Nearest_pixels_to_stations4/TWI/nearest_values_TWI.csv")
srtmMtpi_df = pd.read_csv("Nearest_pixels_to_stations4/srtmMtpiGreece/nearest_values_srtmMtpiGreece.csv")
topoDiversity_df = pd.read_csv("Nearest_pixels_to_stations4/topoDiversity/nearest_values_topoDiversity.csv")
landforms_df = pd.read_csv("Nearest_pixels_to_stations4/landformsGreece/nearest_values_landformsGreece.csv")

### <span style="color: teal;">Data Cleaning and Preparation <span>

In [70]:
# Drop unnecessary columns
wind_df = wind_df.drop(columns=["Tif Name"])
tpi_df = tpi_df.drop(columns=["Tif Name", "Date"])

# Extract date from water_vapor_df
water_vapor_df['Date'] = pd.to_datetime(water_vapor_df['Tif Name'].str.extract(r'(\d{4}_\d{2}_\d{2})')[0], format='%Y_%m_%d')

# Ensure numeric values in TWI and replace invalid values with NaN
df_twi["Value TWI"] = pd.to_numeric(df_twi["Value TWI"], errors="coerce")
df_twi["Value TWI"].replace([-99999, -np.inf], np.nan, inplace=True)

# Extract date from lst_df
def extract_date_from_tif_name(tif_name):
    year = int(tif_name[1:5])  # Extract YYYY
    julian_day = int(tif_name[5:8])  # Extract DDD
    date = datetime.strptime(f"{year}-{julian_day}", "%Y-%j").date()
    return date

lst_df['Date'] = lst_df['Tif Name'].apply(extract_date_from_tif_name)
lst_df['Date'] = pd.to_datetime(lst_df['Date'])

# Convert 'Date' columns to datetime64 explicitly in all dataframes
daily_station_summed_IMERG['Date'] = pd.to_datetime(daily_station_summed_IMERG['Date'])
aggregated_df_COT_no_nans['Date'] = pd.to_datetime(aggregated_df_COT_no_nans['Date'])
aggregated_df_CER_no_nans['Date'] = pd.to_datetime(aggregated_df_CER_no_nans['Date'])
aggregated_df_CWP_no_nans['Date'] = pd.to_datetime(aggregated_df_CWP_no_nans['Date'])
wind_df['Date'] = pd.to_datetime(wind_df['Date'])
water_vapor_df['Date'] = pd.to_datetime(water_vapor_df['Date'])

# Filter data to start from 2010-02-16
daily_station_summed_IMERG = daily_station_summed_IMERG[daily_station_summed_IMERG['Date'] >= '2010-02-16']
lst_df = lst_df[lst_df['Date'] >= '2010-02-16']

/tmp/ipykernel_15901/3161060813.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_twi["Value TWI"].replace([-99999, -np.inf], np.nan, inplace=True)


### <span style="color: teal;">Merge MODIS, LST, Water Vapor, and Wind Data <span>

In [71]:
# Merge dataframes
merged_df = aggregated_df_COT_no_nans[['Date', 'Station Name', 'Station LAT', 'Station LON', 'Altitude', 'Value COT']]
merged_df = pd.merge(merged_df, aggregated_df_CER_no_nans[['Date', 'Station Name', 'Value CER']], 
                     on=['Date', 'Station Name'], how='left')
merged_df = pd.merge(merged_df, aggregated_df_CWP_no_nans[['Date', 'Station Name', 'Value CWP']], 
                     on=['Date', 'Station Name'], how='left')
merged_df = pd.merge(merged_df, daily_station_summed_IMERG[['Date', 'Station Name', 'Value IMERG']], 
                     on=['Date', 'Station Name'], how='left')
merged_df = pd.merge(merged_df, water_vapor_df[['Date', 'Station Name', 'Value Water_Vapor_Column']], 
                     on=['Date', 'Station Name'], how='left')
merged_df = pd.merge(merged_df, wind_df[['Date', 'Station Name', 'Band 1','Band 2', 'Band 3', 'Band 4']], 
                     on=['Date', 'Station Name'], how='left')

### <span style="color: teal;">Merge Topographic and LST Data <span>

In [72]:
# Merge static dataframes (no 'Date' column)
merged_df = pd.merge(merged_df, aspect_df[['Station Name', 'Value ASPECT']], 
                     on='Station Name', how='left')
merged_df = pd.merge(merged_df, lst_df[['Date', 'Station Name', 'Value LST']], 
                     on=['Date', 'Station Name'], how='left')
merged_df = pd.merge(merged_df, slope_df[['Station Name', 'Value SLOPE']], 
                     on='Station Name', how='left')
merged_df = pd.merge(merged_df, tpi_df[['Station Name', 'Value TPI']], 
                     on='Station Name', how='left')
merged_df = pd.merge(merged_df, df_twi[['Station Name', 'Value TWI']], 
                     on='Station Name', how='left')

### <span style="color: teal;">Create Intermediate Merged DataFrame <span>

In [73]:
# Create intermediate merged dataframe
MODIS_IMERG_df = merged_df[['Date', 'Station Name', 'Station LAT', 'Station LON', 'Altitude', 
                            'Value COT', 'Value CER', 'Value CWP', 'Value IMERG', 'Band 1',	'Band 2',	'Band 3',	'Band 4',
                            'Value Water_Vapor_Column','Value ASPECT', 'Value TPI', 'Value TWI']]

# Drop duplicates based on 'Date' and 'Station Name'
MODIS_IMERG_df = MODIS_IMERG_df.drop_duplicates(subset=['Date', 'Station Name'])

### <span style="color: teal;"> Merge Water Vapor Data <span>

In [74]:
# Rename the column in water_vapor_df
water_vapor_df = water_vapor_df.rename(columns={'Value Water_Vapor_Column': 'Value WVC'})

# Merge with MODIS_IMERG_df
MODIS_IMERG_ERA_Water_df = pd.merge(MODIS_IMERG_df, 
                          water_vapor_df[['Date', 'Station Name', 'Value WVC']], 
                          on=['Date', 'Station Name'], 
                          how='left')

In [75]:
MODIS_IMERG_ERA_Water_df.head()

,Date,Station Name,Station LAT,Station LON,Altitude,Value COT,Value CER,Value CWP,Value IMERG,Band 1,Band 2,Band 3,Band 4,Value Water_Vapor_Column,Value ASPECT,Value TPI,Value TWI,Value WVC
0,2010-01-01,ΗΡΑΚΛΕΙΟ,35.3353,25.1820,39,110.0,2990.0,20.0,NaN,6.0,3.496991,4.298134,5.0,0.234500,239.064178,0.875,2.908512,0.234500
1,2010-01-01,ΙΩΑΝΝΙΝΑ,39.6898,20.8281,475,1875.5,1386.0,102.0,NaN,6.0,4.217719,4.217719,6.0,0.725667,26.855507,NaN,13.066851,0.725667
2,2010-01-01,ΚΑΛΑΜΑΤΑ,37.0691,22.0226,6,258.0,962.0,16.0,NaN,7.0,3.952173,4.273682,6.0,1.458000,19.030289,2.125,2.083509,1.458000
3,2010-01-01,ΚΑΣΤΕΛΛΙ,35.1888,25.3289,349,110.0,2682.0,18.0,NaN,6.0,4.059007,4.059007,6.0,0.446000,284.147400,NaN,12.768541,0.446000
4,2010-01-01,ΚΕΡΚΥΡΑ,39.6081,19.9139,1,93.0,2918.0,18.0,NaN,7.0,11.584073,11.584073,7.0,1.133333,282.453491,1.625,NaN,1.133333


### <span style="color: teal;">Merge Topographic Data <span>

In [76]:
# Merge topographic data
MODIS_IMERG_ERA_Water_Topo_df = pd.merge(MODIS_IMERG_ERA_Water_df, 
                          srtmMtpi_df[['Station LAT', 'Station Name', 'Value srtmMtpiGreece']], 
                          on=['Station LAT', 'Station Name'], 
                          how='left')
MODIS_IMERG_ERA_Water_Topo2_df = pd.merge(MODIS_IMERG_ERA_Water_Topo_df, 
                          topoDiversity_df[['Station LAT', 'Station Name', 'Value topoDiversity']], 
                          on=['Station LAT', 'Station Name'], 
                          how='left')

MODIS_IMERG_ERA_Water_Topo3_df = pd.merge(MODIS_IMERG_ERA_Water_Topo2_df, 
                          landforms_df[['Station LAT', 'Station Name', 'Value landformsGreece']], 
                          on=['Station LAT', 'Station Name'], 
                          how='left')

In [77]:
MODIS_IMERG_ERA_Water_Topo3_df.head()

,Date,Station Name,Station LAT,Station LON,Altitude,Value COT,Value CER,Value CWP,Value IMERG,Band 1,...,Band 3,Band 4,Value Water_Vapor_Column,Value ASPECT,Value TPI,Value TWI,Value WVC,Value srtmMtpiGreece,Value topoDiversity,Value landformsGreece
0,2010-01-01,ΗΡΑΚΛΕΙΟ,35.3353,25.1820,39,110.0,2990.0,20.0,NaN,6.0,...,4.298134,5.0,0.234500,239.064178,0.875,2.908512,0.234500,2.0,0.173655,21.0
1,2010-01-01,ΙΩΑΝΝΙΝΑ,39.6898,20.8281,475,1875.5,1386.0,102.0,NaN,6.0,...,4.217719,6.0,0.725667,26.855507,NaN,13.066851,0.725667,NaN,0.235327,32.0
2,2010-01-01,ΚΑΛΑΜΑΤΑ,37.0691,22.0226,6,258.0,962.0,16.0,NaN,7.0,...,4.273682,6.0,1.458000,19.030289,2.125,2.083509,1.458000,NaN,0.029642,34.0
3,2010-01-01,ΚΑΣΤΕΛΛΙ,35.1888,25.3289,349,110.0,2682.0,18.0,NaN,6.0,...,4.059007,6.0,0.446000,284.147400,NaN,12.768541,0.446000,NaN,0.135265,34.0
4,2010-01-01,ΚΕΡΚΥΡΑ,39.6081,19.9139,1,93.0,2918.0,18.0,NaN,7.0,...,11.584073,7.0,1.133333,282.453491,1.625,NaN,1.133333,NaN,0.085118,21.0


### <span style="color:teal;">Load and Prepare Precipitation Data from Sations<span>

In [61]:
# Load and prepare precipitation data
df_daily_precip = pd.read_csv('df_daily_precip.csv')
df_daily_precip[['YEAR', 'MONTH', 'DAY']] = df_daily_precip[['YEAR', 'MONTH', 'DAY']].astype(int)
df_daily_precip['Date'] = pd.to_datetime(df_daily_precip[['YEAR', 'MONTH', 'DAY']])
df_daily_precip = df_daily_precip.drop(columns=['YEAR', 'MONTH', 'DAY'])

# Ensure Date columns are in datetime format
df_daily_precip['Date'] = pd.to_datetime(df_daily_precip['Date'])
MODIS_IMERG_ERA_Water_Topo3_df['Date'] = pd.to_datetime(MODIS_IMERG_ERA_Water_Topo3_df['Date'])

# Filter precipitation data to only include dates from 2010 and onwards
df_daily_precip_filtered = df_daily_precip[df_daily_precip['Date'] >= '2010-02-16']

In [62]:
MODIS_IMERG_ERA_Water_Topo3_df.head()

,Date,Station Name,Station LAT,Station LON,Altitude,Value COT,Value CER,Value CWP,Value IMERG,Band 1,...,Band 3,Band 4,Value Water_Vapor_Column,Value ASPECT,Value TPI,Value TWI,Value WVC,Value srtmMtpiGreece,Value topoDiversity,Value landformsGreece
0,2010-01-01,ΗΡΑΚΛΕΙΟ,35.3353,25.1820,39,110.0,2990.0,20.0,NaN,6.0,...,4.298134,5.0,0.234500,239.064178,0.875,2.908512,0.234500,2.0,0.173655,21.0
1,2010-01-01,ΙΩΑΝΝΙΝΑ,39.6898,20.8281,475,1875.5,1386.0,102.0,NaN,6.0,...,4.217719,6.0,0.725667,26.855507,NaN,13.066851,0.725667,NaN,0.235327,32.0
2,2010-01-01,ΚΑΛΑΜΑΤΑ,37.0691,22.0226,6,258.0,962.0,16.0,NaN,7.0,...,4.273682,6.0,1.458000,19.030289,2.125,2.083509,1.458000,NaN,0.029642,34.0
3,2010-01-01,ΚΑΣΤΕΛΛΙ,35.1888,25.3289,349,110.0,2682.0,18.0,NaN,6.0,...,4.059007,6.0,0.446000,284.147400,NaN,12.768541,0.446000,NaN,0.135265,34.0
4,2010-01-01,ΚΕΡΚΥΡΑ,39.6081,19.9139,1,93.0,2918.0,18.0,NaN,7.0,...,11.584073,7.0,1.133333,282.453491,1.625,NaN,1.133333,NaN,0.085118,21.0


### <span style="color: teal;"> Merge Precipitation Data <span>

In [63]:
# Merge the filtered precipitation data with final_df
merged_final_df = pd.merge(MODIS_IMERG_ERA_Water_Topo3_df, 
                           df_daily_precip[['Date', 'Station Name', 'PRECIPITATION']], 
                           on=['Date', 'Station Name'], 
                           how='left')

In [64]:
merged_final_df

,Date,Station Name,Station LAT,Station LON,Altitude,Value COT,Value CER,Value CWP,Value IMERG,Band 1,...,Band 4,Value Water_Vapor_Column,Value ASPECT,Value TPI,Value TWI,Value WVC,Value srtmMtpiGreece,Value topoDiversity,Value landformsGreece,PRECIPITATION
0,2010-01-01,ΗΡΑΚΛΕΙΟ,35.3353,25.1820,39,110.0,2990.0,20.0,NaN,6.0,...,5.0,0.234500,239.064178,0.875,2.908512,0.234500,2.0,0.173655,21.0,NaN
1,2010-01-01,ΙΩΑΝΝΙΝΑ,39.6898,20.8281,475,1875.5,1386.0,102.0,NaN,6.0,...,6.0,0.725667,26.855507,NaN,13.066851,0.725667,NaN,0.235327,32.0,NaN
2,2010-01-01,ΚΑΛΑΜΑΤΑ,37.0691,22.0226,6,258.0,962.0,16.0,NaN,7.0,...,6.0,1.458000,19.030289,2.125,2.083509,1.458000,NaN,0.029642,34.0,NaN
3,2010-01-01,ΚΑΣΤΕΛΛΙ,35.1888,25.3289,349,110.0,2682.0,18.0,NaN,6.0,...,6.0,0.446000,284.147400,NaN,12.768541,0.446000,NaN,0.135265,34.0,NaN
4,2010-01-01,ΚΕΡΚΥΡΑ,39.6081,19.9139,1,93.0,2918.0,18.0,NaN,7.0,...,7.0,1.133333,282.453491,1.625,NaN,1.133333,NaN,0.085118,21.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53014,2023-12-31,ΡΟΔΟΣ,36.4022,28.0882,7,324.0,1999.0,42.0,0.0,8.0,...,3.0,0.597000,216.274841,NaN,9.615806,0.597000,NaN,0.416559,31.0,NaN
53015,2023-12-31,ΣΚΥΡΟΣ,38.9631,24.4905,21,303.5,3673.0,68.5,0.0,NaN,...,NaN,0.154500,137.879257,1.500,4.831018,0.154500,NaN,0.176374,21.0,NaN
53016,2023-12-31,ΣΠΑΤΑ (ΒΕΝΙΖΕΛΟΣ),37.9206,23.9314,72,109.0,4690.0,31.0,15.0,8.0,...,1.0,0.480000,64.123985,0.875,11.225244,0.480000,NaN,0.070582,21.0,NaN
53017,2023-12-31,ΤΑΝΑΓΡΑ,38.3354,23.5630,140,76.5,5057.0,23.5,13.0,8.0,...,4.0,0.572667,214.422577,1.750,11.870600,0.572667,NaN,0.116138,34.0,NaN


### <span style="color:teal;">Save Final Merged DataFrame <span>

In [66]:
# Save the final merged dataframe to CSV
merged_final_df.to_csv('merged_final_df_ML2.csv', index=False)